## Filesystem Mounting
Process of attaching disk, partitions, usb drives, network drive onto the Linux directory tree. As an example, lets say we have a `/` directory as:

```
/
├── home/
├── etc/
└── mnt/
```

After we mount a USB drive at `/mnt/usb`:
```
/
├── home/
├── etc/
└── mnt/
    └── usb/
        ├── photo.jpg
        └── document.txt
```

### Key Terms
**Device:**	the physical or virtual storage unit (`/dev/sda1`, a USB stick, a network share)  

**Filesystem:**	the format the data is organized in on that device (`ext4`, `NTFS`, `exFAT`, etc.)

**Mount point:** an existing, empty directory that becomes the  entry door into that device's contents

**Mounting:** tThe act of attaching a device's filesystem to a mount point

**Unmounting:**	the reverse process of mounting, the mount point directory becomes empty

## Filesystems
A filesystem defines how raw bytes on a block device translate to files, directories, permissions, metadata and more. Some common filesystems:
- `ext4`: default on most distributions (like Debian ones)
- `xfs`: default on Fedora and related distributions
- `zfs`
- `ntfs`: used by Windows
- `tmpfs`: RAM based

## Device and Partitions
Storage devices are divided into different partitions and each partition is listed under `/dev`.
- `/dev/sdX`: SATA SSDs, HDDs, USB flash drives, USB external HDDs are all represented in this format.  
  ```
  /dev/sda    # first detected disk (whole disk, unpartitioned reference)
  /dev/sda1   # first partition on that disk
  /dev/sda2   # second partition on that disk
  /dev/sdb    # second detected disk
  /dev/sdb1   # first partition on the second disk
  ```  
- `/dev/nvmeXnYpZ`: modern M.2 SSDs use different naming structure  
  ```
  /dev/nvme0n1       # first NVMe controller (nvme0), first namespace (n1) — the whole disk
  /dev/nvme0n1p1     # first partition (p1) on that namespace
  /dev/nvme0n1p2     # second partition
  /dev/nvme1n1       # second physical NVMe drive, if present
  ```  
- `/dev/vdX`: virtual disks, used inside virtual machines
  ```
  /dev/vda      # first virtual disk
  /dev/vda1     # first partition
  ```

Different partitions can have different filesystem. To view the devices and partitions we can:
```sh
$ lsblk
NAME        MAJ:MIN RM   SIZE RO TYPE MOUNTPOINTS
sda           8:0    0 931.5G  0 disk 
├─sda1        8:1    0    16M  0 part 
└─sda2        8:2    0 931.5G  0 part /media/data
nvme0n1     259:0    0 238.5G  0 disk 
├─nvme0n1p1 259:1    0   260M  0 part /boot/efi
├─nvme0n1p2 259:2    0    16M  0 part 
├─nvme0n1p3 259:3    0   100G  0 part 
├─nvme0n1p4 259:4    0  1000M  0 part 
└─nvme0n1p5 259:5    0 137.2G  0 part /

```

### Parition Table
Before any partition can exist, the disk needs a partition table which is a data structure present at the very start of the disk that records how many partitions exist, their locations, etc. There are two partition table formats:
- MBR (Master Boot Record): this is the older standard taking up the firsst 512 bytes. Due this size constraint you can only define 4 partitions. But there is a workaround as shown below:
  ```
  /dev/sda1   # primary
  /dev/sda2   # primary
  /dev/sda3   # extended acts as container, not directly usable
    ├─ /dev/sda5   logical # can't name sda4, it is reserved for 4th primary slot
    ├─ /dev/sda6   logical
    └─ /dev/sda7   logical
  ```
  MBR has other limitations:
  - maximum supported disk size is 2TB
  - no redundancy for the first 512 bytes

- GPT (Guided Partition Table): is the newer format which overcomes the limitations of MBR.

To check the parition table format, we can:
```sh
$ sudo parted /dev/nvme0n1 print
Model: SKHynix_HFS256GD9TNG-L3A0B (nvme)
Disk /dev/nvme0n1: 256GB
Sector size (logical/physical): 512B/512B
Partition Table: gpt
Disk Flags: 

Number  Start   End    Size    File system  Name                          Flags
 1      1049kB  274MB  273MB   fat32        EFI system partition          boot, esp, no_automount
 2      274MB   290MB  16.8MB               Microsoft reserved partition  msftres, no_automount
 3      290MB   108GB  107GB   ntfs         Basic data partition          msftdata
 5      108GB   255GB  147GB   ext4
 4      255GB   256GB  1049MB  ntfs         Basic data partition          hidden, diag, no_automount
```

## Shadowing
If the mount point directory already contains some files and we mount something there - the original contents are hidden. As an example, lets say we have:
```sh
$ ls /mnt/data
oldfile.txt

# After mounting at /mnt/data
$ ls /mnt/data
newfile.txt

# After unmounting
$ ls /mnt/data
oldfile.txt
```

## `mount` Commands
To mount a device at a mount point, we use the `mount` command:
```sh
# /mnt/usb must already exist
$ mount /dev/sdb1 /mnt/usb
```

We can choose to keep mounts read-only or read-write (the default) by specifying different flags:
```sh
$ sudo mount -o ro /dev/sdb1 /mnt/usb          # read-only, -rw for read write
                                               # -noexec prevents executing binaries from this mount
```

To view currently mounted items, we can use `mount` command again (without any options) or better use `findmnt` which gives better formatted result:
```sh
$ findmnt
TARGET     SOURCE      FSTYPE OPTIONS
/          /dev/sda2   ext4   rw,relatime
├─/boot    /dev/sda1   vfat   rw,relatime
└─/mnt/usb /dev/sdb1   exfat  rw,relatime
```

To unmount we can specify the mount point or the device:
```sh
$ sudo unmount /mnt/usb
# Or
$ sudo unmount /dev/sdb1
```

## `fstab`
Mount points set using the `mount` command are all temporary, it goes away on reboot. `/etc/fstab` is a configuration file that defines which filesystems get mounted automatically on boot.

```sh
$ cat /etc/fstab
# /etc/fstab: static file system information.
#
# Use 'blkid' to print the universally unique identifier for a
# device; this may be used with UUID= as a more robust way to name devices
# that works even if disks are added and removed. See fstab(5).
#
# <file system> <mount point>   <type>  <options>       <dump>  <pass>
# / was on /dev/nvme0n1p5 during curtin installation
/dev/disk/by-uuid/8caa7dcb-a6d3-42cd-8b9e-dbb0a0c5c868 / ext4 defaults 0 1
# /boot/efi was on /dev/nvme0n1p1 during curtin installation
/dev/disk/by-uuid/4A0E-2B36 /boot/efi vfat defaults 0 1
/swap.img	none	swap	sw	0	0
UUID=A258D6A658D6790D /media/data ntfs uid=1000,gid=1000,rw,user,exec,umask=000 0 0
```

Typically, instead of using the partition path (like `/dev/sda1`), we use its UUID (since UUID remains constant, whereas path can change). To know the UUID, use:
```sh
$ sudo blkid /dev/sda2
/dev/sda2: LABEL="Data" BLOCK_SIZE="512" UUID="A258D6A658D6790D" TYPE="ntfs" PARTLABEL="Basic data partition" PARTUUID="e8435959-6a7f-4433-98f7-a35ea3d7d3a9"
```

Do `findmnt --verify` to ensure the entry is correct. 